# 07 Follow-up 1 for model E: does forcing race out of z close the gap?

Plan written before the run: `docs/FOLLOWUP_E_FORCE_RACE.md` (observation, three sentences, prediction, design, reading rules). Exploratory: unadjusted paired CIs, no Holm, no success verdict. 8 fixed settings (lambda 0 control; base-rate target lambda 0.1 to 10; max-error target lambda 1, 10), 60 epochs each, no tuning, leakage probe on z. GPU: T4, estimate 30-60 min. Charts only; no images are opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, subprocess, zipfile, hashlib
os.environ['CRF_ENV'] = 'colab'
REPO = pathlib.Path('/content/robust_child_filtering_model')
URL = 'github.com/isaacmutuma/robust_child_filtering_model.git'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/robust_child_filtering_model')
os.chdir('/content')
!rm -rf {REPO}
# Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar; fine-grained token,
# read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU works, just slower)')

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def done(stem):
    """True if Drive holds a clean manifest for `stem` and its npz still matches it."""
    m = DRIVE_ROOT / 'results' / 'embeddings' / f'{stem}.json'
    if not m.exists(): return False
    d = json.loads(m.read_text())
    npz = DRIVE_ROOT / d['npz']['drive_path']
    return d['git']['dirty'] is False and npz.exists() and sha256(npz) == d['npz']['sha256']

def save_manifest(rel):
    """Copy a results JSON to Drive after checking it came from clean, committed code."""
    src = REPO / 'results' / rel
    d = json.loads(src.read_text())
    assert d['git']['dirty'] is False, 'code was modified in this runtime'
    dst = DRIVE_ROOT / 'results' / rel; dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(src.read_text()); print('saved', rel, 'commit', d['git']['commit'][:7])
    return d

## 1. Preconditions

In [ ]:
A = json.loads((DRIVE_ROOT / 'results/models/A.json').read_text())
assert A['git']['dirty'] is False
assert sha256(DRIVE_ROOT / A['oof_file']['drive_path']) == A['oof_file']['sha256'], "A's OOF file changed"
assert done('utkface__clip_vitl14'), 'UTKFace embeddings missing or changed on Drive'
E = json.loads((DRIVE_ROOT / 'results/models/E_inv.json').read_text())
print('E_inv reference: probe', round(E['leakage_probe']['bal_acc']['model']['est'], 3), 'A z', round(E['leakage_probe']['bal_acc']['A']['est'], 3),
      'gap', round(E['report']['tpr_gap_W_minus_B']['est'], 3))
print(open('docs/FOLLOWUP_E_FORCE_RACE.md').read().split('## Design')[0])

## 2. Run the sweep

In [ ]:
!python scripts/followup_e_force_race.py
d = save_manifest('followups/E_force_race.json')

## 3. Dose-response table

In [ ]:
print(f"{'setting':10s} {'probe':>6s} {'gap':>6s} {'gap CI':>16s} {'red. vs A':>10s} {'MAE':>5s} {'age-err B-W':>11s} {'teen B/W':>11s} {'max FPR':>7s} removed")
for n, r in d['settings'].items():
    rep, g = r['report'], r['report']['tpr_gap_W_minus_B']
    bg = rep['by_group']
    fpr = max(x['fpr_adult']['est'] for x in bg.values())
    print(f"{n:10s} {r['leakage_probe']['bal_acc']['model']['est']:6.3f} {g['est']:6.3f} [{g['ci'][0]:.3f}, {g['ci'][1]:.3f}] "
          f"{r['paired_vs_A']['gap_reduction_A_minus_M']['est']:+10.3f} {rep['mae_overall']['est']:5.2f} {rep['age_err_gap_B_minus_W']['est']:+11.2f} "
          f"{bg['Black']['tpr_teen']['est']:.2f}/{bg['White']['tpr_teen']['est']:.2f} {fpr:7.3f} {r['race_removed']}")

## 4. Charts: race leakage and gap against lambda (base-rate target)

In [ ]:
import matplotlib.pyplot as plt
pr = [n for n in d['settings'] if n.startswith('prior')]
lam = [d['settings'][n]['setting']['adv_lambda'] for n in pr]
probe = [d['settings'][n]['leakage_probe']['bal_acc']['model'] for n in pr]
gap = [d['settings'][n]['report']['tpr_gap_W_minus_B'] for n in pr]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a_, vals, lab in [(ax[0], probe, 'probe balanced accuracy'), (ax[1], gap, 'White - Black TPR gap')]:
    a_.errorbar(lam, [v['est'] for v in vals], yerr=[[v['est'] - v['ci'][0] for v in vals], [v['ci'][1] - v['est'] for v in vals]], fmt='o-', capsize=3)
    a_.set_xscale('log'); a_.set_xlabel('lambda (base-rate target)'); a_.set_ylabel(lab)
ax[0].axhline(0.2, ls='--', c='grey', lw=1); ax[0].axhline(E['leakage_probe']['bal_acc']['A']['est'], ls=':', c='k', lw=1)
ax[1].axhline(A['report']['tpr_gap_W_minus_B']['est'], ls=':', c='k', lw=1); ax[1].axhline(0, ls='--', c='grey', lw=1)
plt.tight_layout(); plt.show()

## 5. Aggregate output for the repo (JSON only)

In [ ]:
out = DRIVE_ROOT / 'step11_E_followup_outputs.zip'
with zipfile.ZipFile(out, 'w') as z:
    z.write(DRIVE_ROOT / 'results/followups/E_force_race.json', 'results/followups/E_force_race.json')
print(out, out.stat().st_size, 'bytes, sha256', sha256(out))